# 🎓 Capstone: The Full EPS Research Picture
### EPS Research High-School Exploration Track — Ages 15-18

You've completed Track B. This capstone summarizes several analyses using
the corrected FAIR² implementation.

The local SPARC result below uses the same frozen **84-galaxy cohort** as
the published analysis. Omega values are reported in rad/Gyr, while
velocity arithmetic uses the native km/s/kpc form.

Earlier FAIR² material described a cross-epoch omega sign reversal.
That interpretation resulted from an incorrect Eq. 6 grouping and is
**not part of the corrected result**. Under the corrected canonical
equation, the high-redshift result does not establish that sign reversal.

This capstone therefore presents the local frozen-cohort result without
making a cross-epoch sign-reversal claim.

In [ ]:
# ── Colab setup: canonical FAIR² corpus paths ─────────────
import os, sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    import urllib.request
    CORPORA = {
        'rotation_curve_corpus_v7.json': 'https://zenodo.org/records/19563417/files/rotation_curve_corpus_v7.json',
        'high_z_kinematic_corpus_Z1.json': 'https://zenodo.org/records/21834678/files/high_z_kinematic_corpus_Z1.json',
        'dwarf_irregular_corpus_v1.json': 'https://zenodo.org/records/20320362/files/dwarf_irregular_corpus_v1.json',
    }
    for filename, url in CORPORA.items():
        if not os.path.exists(filename):
            print(f"Downloading {filename}...")
            urllib.request.urlretrieve(url, filename)
            print(f"  ✓ {filename}")
        else:
            print(f"  Already present: {filename}")

    HI_PATH = 'rotation_curve_corpus_v7.json'
    Z1_PATH = 'high_z_kinematic_corpus_Z1.json'
    DWARF_PATH = 'dwarf_irregular_corpus_v1.json'
    print("Ready.")
else:
    HI_PATH = '../hi/rotation_curve_corpus_v7.json'
    Z1_PATH = '../highz/high_z_kinematic_corpus_Z1.json'
    DWARF_PATH = '../dwarfs/dwarf_irregular_corpus_v1.json'
    print("Running locally — using canonical repository corpus paths.")


In [ ]:
import matplotlib
matplotlib.use('Agg')
import json, numpy as np, matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Frozen published cohort: (canonical SPARC galaxy ID, omega in rad/Gyr)
FROZEN_84 = [('NGC3741', 7.032),
 ('UGC08550', 9.317),
 ('NGC3109', 10.244),
 ('UGC07603', 14.2),
 ('DDO064', 15.35),
 ('UGC01281', 11.37),
 ('UGC07151', 12.53),
 ('UGC07399', 14.86),
 ('UGC04278', 13.76),
 ('NGC3972', 13.93),
 ('NGC7793', 11.4),
 ('F571-8', 9.17),
 ('UGC05721', 11.51),
 ('UGC07323', 13.41),
 ('NGC3521', 10.25),
 ('F563-V2', 11.08),
 ('ESO116-G012', 11.02),
 ('F568-1', 10.52),
 ('ESO079-G014', 10.37),
 ('NGC3893', 6.47),
 ('UGC08286', 9.82),
 ('NGC0024', 9.55),
 ('NGC0100', 9.37),
 ('NGC0891', 8.99),
 ('NGC4217', 9.99),
 ('UGC06917', 8.3),
 ('NGC7814', 8.52),
 ('NGC3917', 8.82),
 ('F583-4', 9.33),
 ('IC4202', 9.3),
 ('UGC08490', 6.95),
 ('NGC4088', 6.98),
 ('NGC6946', 6.83),
 ('F568-3', 6.54),
 ('F568-V1', 6.55),
 ('NGC2403', 6.32),
 ('UGC12632', 6.27),
 ('UGC11455', 6.24),
 ('NGC5985', 6.16),
 ('UGC00731', 5.99),
 ('UGC06786', 5.87),
 ('NGC6195', 5.83),
 ('UGC12732', 5.81),
 ('NGC4157', 5.46),
 ('UGC06930', 5.44),
 ('UGC03205', 5.27),
 ('UGC11820', 5.21),
 ('NGC4100', 5.2),
 ('UGC03546', 5.29),
 ('F563-1', 5.02),
 ('NGC4559', 5.3),
 ('F583-1', 5.16),
 ('NGC2955', 5.71),
 ('NGC7331', 4.9),
 ('NGC4183', 4.92),
 ('DDO161', 4.69),
 ('NGC6503', 4.3),
 ('NGC2998', 4.61),
 ('NGC1090', 5.24),
 ('NGC5033', 3.79),
 ('NGC5371', 3.78),
 ('F579-V1', 7.13),
 ('UGC06983', 5.74),
 ('NGC2841', 3.58),
 ('UGC05750', 3.44),
 ('UGC05005', 3.38),
 ('UGC02885', 3.4),
 ('NGC5055', 2.89),
 ('NGC6674', 2.81),
 ('UGC01230', 2.74),
 ('UGC06614', 2.49),
 ('UGC02487', 2.49),
 ('UGC00128', 2.23),
 ('NGC0801', 3.32),
 ('UGC09133', 1.97),
 ('UGC07125', 3.09),
 ('NGC1003', 3.49),
 ('NGC3198', 3.33),
 ('NGC2903', 7.01),
 ('ESO563-G021', 7.31),
 ('NGC5585', 8.06),
 ('F574-1', 7.68),
 ('UGC06446', 7.11),
 ('UGC07524', 7.15)]

assert len(FROZEN_84)==84
assert len({name for name,omega in FROZEN_84})==84

with open(HI_PATH) as f:
    corpus=json.load(f)

sparc={
    g['galaxy']:g
    for g in corpus['galaxies']
    if g.get('survey')=='SPARC'
}

omegas=[]
vmaxs=[]
rmaxs=[]
missing=[]

for name,omega_rad_gyr in FROZEN_84:
    g=sparc.get(name)
    if g is None:
        missing.append(name)
        continue

    d=[
        p for p in g.get('data',[])
        if p.get('Rad',0)>0 and p.get('Vobs',0)>0
    ]
    if len(d)<2:
        missing.append(name)
        continue

    omegas.append(omega_rad_gyr)
    vmaxs.append(max(p['Vobs'] for p in d))
    rmaxs.append(d[-1]['Rad'])

assert not missing, f"Frozen cohort missing/invalid: {missing}"
assert len(omegas)==84

omegas=np.array(omegas)
vmaxs=np.array(vmaxs)
rmaxs=np.array(rmaxs)

# Example NGC2403 rotation curve, canonical SPARC identity only.
g_ex=sparc['NGC2403']
d_ex=[
    p for p in g_ex['data']
    if p.get('Rad',0)>0 and p.get('Vobs',0)>0
]
R_ex=np.array([p['Rad'] for p in d_ex])
Vobs_ex=np.array([p['Vobs'] for p in d_ex])

R1e,V1e=R_ex[0],Vobs_ex[0]
R2e,V2e=R_ex[-1],Vobs_ex[-1]

# LOCKED Eq. 6 PRECEDENCE:
outer_term    = (V2e / R2e)
inner_term    = (V1e / R1e) * ((R1e / R2e) ** 1.5)
omega_kms_kpc = outer_term - inner_term
omega_rad_gyr = omega_kms_kpc * 1.0227

# LOCKED UNIT INVARIANT:
V_adj = Vobs_ex - R_ex * omega_kms_kpc

fig=plt.figure(figsize=(13,8))
gs=gridspec.GridSpec(2,3,figure=fig,hspace=0.4,wspace=0.35)

ax1=fig.add_subplot(gs[0,0])
ax1.plot(R_ex,Vobs_ex,'ko-',ms=5,lw=1.5,label='Observed')
ax1.plot(
    R_ex,V_adj,'b-',lw=2,
    label=f'ω-corrected (ω={omega_rad_gyr:+.2f} rad/Gyr)'
)
ax1.set_xlabel('R (kpc)',fontsize=10)
ax1.set_ylabel('V (km/s)',fontsize=10)
ax1.set_title('NGC2403 Rotation Curve',fontsize=10)
ax1.legend(fontsize=8)
ax1.grid(alpha=0.3)

ax2=fig.add_subplot(gs[0,1])
ax2.hist(omegas,bins=20,alpha=0.8,edgecolor='white')
ax2.axvline(
    np.mean(omegas),lw=2,ls='--',
    label=f'Mean {np.mean(omegas):+.2f}'
)
ax2.set_xlabel('ω (rad/Gyr)',fontsize=10)
ax2.set_ylabel('N',fontsize=10)
ax2.set_title('Omega Distribution\\nFrozen N=84 cohort',fontsize=10)
ax2.legend(fontsize=8)

ax3=fig.add_subplot(gs[0,2])
ax3.scatter(vmaxs,omegas,s=15,alpha=0.5)
ax3.axhline(np.mean(omegas),lw=1.5,ls='--')
ax3.set_xlabel('Vmax (km/s)',fontsize=10)
ax3.set_ylabel('ω (rad/Gyr)',fontsize=10)
ax3.set_title('Omega vs Rotation Speed',fontsize=10)
ax3.grid(alpha=0.3)

ax4=fig.add_subplot(gs[1,:])
ax4.axis('off')

summary=[
    ['Statistic','Value','Meaning'],
    ['N galaxies',str(len(omegas)),'Frozen published SPARC cohort'],
    ['Mean ω',f'{np.mean(omegas):+.3f} rad/Gyr','Average stored correction'],
    ['Median ω',f'{np.median(omegas):+.3f} rad/Gyr','Typical value'],
    ['Std ω',f'{np.std(omegas):.3f} rad/Gyr','Galaxy-to-galaxy variation'],
    ['All positive',str(bool(np.all(omegas>0))),'Sign in frozen local cohort'],
    ['Median Vmax',f'{np.median(vmaxs):.1f} km/s','Typical rotation speed'],
    ['Median Rmax',f'{np.median(rmaxs):.1f} kpc','Typical outer radius'],
]

tbl=ax4.table(
    cellText=summary[1:],
    colLabels=summary[0],
    loc='center',
    cellLoc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.auto_set_column_width([0,1,2])

ax4.set_title(
    'Summary: Corrected FAIR² Local SPARC Result',
    fontsize=11,pad=20
)

plt.suptitle(
    '🎓 Capstone — Corrected FAIR² Analysis\\n'
    'Frozen N=84 SPARC cohort | canonical Eq. 6',
    fontsize=12
)

plt.savefig('hs_b_10_capstone.png',dpi=150,bbox_inches='tight')
plt.show()

print('Capstone complete!')
print(f'Frozen cohort analyzed: {len(omegas)} galaxies')
print(f'Mean omega = {np.mean(omegas):+.3f} rad/Gyr')
print(f'All omega values positive in frozen local cohort: {bool(np.all(omegas>0))}')
print('No cross-epoch sign-reversal conclusion is asserted.')
